# Lista 05 - Regras de associacao no Titanic

Execute as celulas em ordem com `titanic completo.csv` no ambiente do Colab. O notebook usa a biblioteca `apyori`, como o `Apriori.ipynb` da disciplina.


## Bibliotecas

A primeira celula instala `apyori`, como no notebook de exemplo da disciplina.


In [ ]:
!pip -q install apyori


## 1. Pré-processamento e codificação

A base tem 1.309 passageiros. Idade (263 faltantes) e tarifa (1) recebem a mediana; embarque (2) recebe a moda. Nome, ticket, cabine e destino são identificadores ou texto de alta cardinalidade. Barco e corpo são informação posterior ao desfecho e gerariam vazamento.

As faixas de idade (0–17, 18–22, 23–40 e 41+) separam crianças, jovens e adultos. As de tarifa (até 10, 10–30, 30–80 e acima de 80) distinguem níveis de pagamento sem criar valores numéricos únicos.

Cada categoria vira um item `Atributo=valor`; cada outra categoria do mesmo atributo vira `NÃO Atributo=valor`. O desfecho recebe apenas o item presente, para evitar tautologias.


In [ ]:
from apyori import apriori

import numpy as np
import pandas as pd


CONFIGURACOES = [(0.15, 0.70), (0.20, 0.75), (0.25, 0.80),
                 (0.30, 0.80), (0.20, 0.70)]
ESCOLHIDA = (0.20, 0.75)


def preparar_base(caminho="titanic completo.csv"):
    original = pd.read_csv(caminho)
    dados = pd.DataFrame(index=original.index)
    dados["Classe"] = original.pclass.map({1: "1a", 2: "2a", 3: "3a"})
    dados["Sexo"] = original.sex.map({"female": "Feminino", "male": "Masculino"})
    idade = original.age.fillna(original.age.median())
    tarifa = original.fare.fillna(original.fare.median())
    embarque = original.embarked.fillna(original.embarked.mode()[0])
    dados["Idade"] = pd.cut(idade, [-np.inf, 17, 22, 40, np.inf],
                            labels=["0-17", "18-22", "23-40", "41+"]).astype(str)
    dados["Tarifa"] = pd.cut(tarifa, [-np.inf, 10, 30, 80, np.inf],
                             labels=["0-10", "10-30", "30-80", "80+"]).astype(str)
    dados["Embarque"] = embarque.map({"C": "Cherbourg", "Q": "Queenstown",
                                       "S": "Southampton"})
    dados["Irmãos/cônjuges"] = pd.cut(original.sibsp, [-1, 0, 1, np.inf],
                                      labels=["0", "1", "2+"]).astype(str)
    dados["Pais/filhos"] = pd.cut(original.parch, [-1, 0, 1, np.inf],
                                  labels=["0", "1", "2+"]).astype(str)
    dados["Sobreviveu"] = original.survived.map({0: "Não", 1: "Sim"})
    assert not dados.isna().any().any()
    assert not (dados == "nan").any().any()
    return original, dados



In [ ]:
original, dados = preparar_base()
print('Dimensoes:', original.shape)
print('Ausencias apos tratamento:', int(dados.isna().sum().sum()))
display(original.head())
display(dados.head())


## 2. Transacoes com presenca e ausencia

Cada passageiro vira uma lista de itens, no mesmo formato aceito por `apyori.apriori`. Itens `NÃO` descrevem as categorias ausentes; para o resultado, usamos apenas `Sobreviveu=Sim` ou `Sobreviveu=Não`.


In [ ]:
def codificar_itens(dados):
    """Um item de presença e complementos de todas as outras categorias.

    Não geramos complementos para o desfecho, pois seriam equivalentes ao
    desfecho oposto e produziriam regras tautológicas.
    """
    itens = {}
    origem = {}
    for atributo in dados.columns.drop("Sobreviveu"):
        for valor in sorted(dados[atributo].unique()):
            nome = f"{atributo}={valor}"
            itens[nome] = dados[atributo].eq(valor).to_numpy()
            origem[nome] = atributo
            nome_ausencia = f"NÃO {atributo}={valor}"
            itens[nome_ausencia] = dados[atributo].ne(valor).to_numpy()
            origem[nome_ausencia] = atributo
    return itens, origem



In [ ]:
def criar_transacoes(dados, itens):
    # Cada linha recebe itens de presenca e ausencia dos preditores.
    transacoes = []
    for i in range(len(dados)):
        transacao = [nome for nome, mascara in itens.items() if mascara[i]]
        transacao.append(f"Sobreviveu={dados['Sobreviveu'].iloc[i]}")
        transacoes.append(transacao)
    return transacoes


itens, origem = codificar_itens(dados)
transacoes = criar_transacoes(dados, itens)
print('Transacoes:', len(transacoes), '| itens preditores:', len(itens))
print('Primeira transacao:')
display(pd.DataFrame({'Item': transacoes[0]}))


## 3. Apriori e comparacao de parametros

A chamada `apriori(transacoes, ...)` e a leitura de `saida` seguem o notebook de exemplo. A busca usa os menores limites testados; os demais limites sao aplicados as regras retornadas. O tamanho maximo do itemset e 3 para manter antecedentes de ate duas caracteristicas.


In [ ]:
def extrair_regras(saida, dados, itens, origem):
    # Mesma leitura de saida do Apriori.ipynb: itemset, regras e metricas.
    linhas = []
    for resultado in saida:
        for estatistica in resultado.ordered_statistics:
            antecedente = sorted(estatistica.items_base)
            consequente = sorted(estatistica.items_add)

            # A questao pede regras cujo consequente e apenas a sobrevivencia.
            if consequente not in (["Sobreviveu=Sim"], ["Sobreviveu=Não"]):
                continue
            if len(antecedente) not in (1, 2):
                continue
            if any(item not in origem for item in antecedente):
                continue
            if len({origem[item] for item in antecedente}) != len(antecedente):
                continue

            mascara = np.logical_and.reduce([itens[item] for item in antecedente])
            resultado_nome = consequente[0].split("=", 1)[1]
            casos = int(np.count_nonzero(mascara & dados["Sobreviveu"].eq(resultado_nome).to_numpy()))
            linhas.append({
                "Antecedente": " E ".join(antecedente),
                "Consequente": consequente[0],
                "Suporte": resultado.support,
                "Confiança": estatistica.confidence,
                "Lift": estatistica.lift,
                "Casos": casos,
                "Total do antecedente": int(mascara.sum()),
                "Itens no antecedente": len(antecedente),
                "Tem ausência": any(item.startswith("NÃO ") for item in antecedente),
                "Número de ausências": sum(item.startswith("NÃO ") for item in antecedente),
                "Padrão": mascara.tobytes(),
            })
    return pd.DataFrame(linhas)


def selecionar_regras(candidatas, suporte, confianca):
    regras = candidatas.loc[
        (candidatas["Suporte"] >= suporte)
        & (candidatas["Confiança"] >= confianca)
        & (candidatas["Lift"] > 1.0)
    ].copy()
    # Em atributos binários, "NÃO Sexo=Masculino" e "Sexo=Feminino" cobrem
    # exatamente as mesmas pessoas. Contamos cada padrão uma só vez.
    regras = regras.sort_values(["Itens no antecedente", "Número de ausências",
                                 "Antecedente"])
    regras = regras.drop_duplicates(["Padrão", "Consequente"])
    return regras.sort_values(["Lift", "Confiança", "Suporte", "Antecedente"],
                              ascending=[False, False, False, True]).reset_index(drop=True)



In [ ]:
suporte_base = min(s for s, _ in CONFIGURACOES)
confianca_base = min(c for _, c in CONFIGURACOES)

# A chamada abaixo segue o Apriori.ipynb da disciplina.
regras_apriori = apriori(transacoes, min_support=suporte_base,
                        min_confidence=confianca_base, min_lift=1.0,
                        max_length=3)
saida = list(regras_apriori)
print('Itemsets retornados pelo apyori:', len(saida))

candidatas = extrair_regras(saida, dados, itens, origem)
comparacao_linhas = []
for suporte_min, confianca_min in CONFIGURACOES:
    encontradas = selecionar_regras(candidatas, suporte_min, confianca_min)
    comparacao_linhas.append({
        'Suporte minimo': suporte_min,
        'Confianca minima': confianca_min,
        'Regras (lift > 1)': len(encontradas),
        'Sobreviveu=Sim': int((encontradas['Consequente'] == 'Sobreviveu=Sim').sum()),
        'Sobreviveu=Nao': int((encontradas['Consequente'] == 'Sobreviveu=Não').sum()),
    })
comparacao = pd.DataFrame(comparacao_linhas)
display(comparacao)
regras = selecionar_regras(candidatas, *ESCOLHIDA)
print('Configuracao escolhida:', ESCOLHIDA, '| regras:', len(regras))


## 4. Regras ordenadas e separadas por desfecho

Suporte é `casos da regra / 1309`; confiança é `casos da regra / casos do antecedente`; lift é `confiança / frequência do desfecho`. Lift acima de 1 indica associação positiva.


In [ ]:
colunas = ['Antecedente', 'Consequente', 'Casos', 'Total do antecedente',
           'Suporte', 'Confiança', 'Lift']
for resultado in ('Sim', 'Não'):
    grupo = regras[regras['Consequente'] == f'Sobreviveu={resultado}']
    print(f'Sobreviveu={resultado}: {len(grupo)} regras')
    display(grupo[colunas].head(15).round(3))


## 5. Regras com itens de ausencia

Esta tabela facilita a captura das regras que contem `NÃO` no antecedente. As interpretacoes escritas estao no arquivo TXT.


In [ ]:
regras_ausencia = regras[regras['Tem ausência']]
print('Regras com ausencia:', len(regras_ausencia))
display(regras_ausencia[colunas].head(15).round(3))
